# E002-A GPU Word TF-IDF Probe

**Experiment:** E002-A  
**Purpose:** Determine GPU viability of global Name Word TF-IDF retrieval.  

**Status Context:**
- E002-A (CPU SciPy) Ã¢â€ â€™ rejected
- E002-B (CPU sparse_dot_topn) Ã¢â€ â€™ rejected operationally
- E002-A Ã¢â€ â€™ current GPU probe

**Architecture Contract:**
- Architecture remains unchanged from the repository implementation.
- No E002 / BM25 / Word TF-IDF / matcher code.
- Notebook is ONLY a launcher. Heavy compute must run on GPU.

## 1. Kaggle GPU Check
Checking environment to ensure GPU is available.

In [ ]:
import sys
import os

print(f"Python Version: {sys.version}")
print(f"Current working directory: {os.getcwd()}")

!nvidia-smi

try:
    import cupy
    print("\nCuPy Version:", cupy.__version__)
    cupy.show_config()
except ImportError:
    print("\nCuPy is not installed.")

## 2. Repository Setup
Cloning the repository if it's not present (e.g. running as a fresh Kaggle notebook).

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/yugtheguy/amazon_ml.git"
REPO_DIR = "/kaggle/working/amazon_ml"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository into {REPO_DIR}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at {REPO_DIR}.")

os.chdir(REPO_DIR)
print("\nChanged directory to repo root.")
print("\n--- Git Status ---")
!git status --short
print("\n--- Git Commit ---")
!git rev-parse HEAD

## 3. Discover Kaggle Input Data
Finding the actual dataset directory securely rather than assuming paths.

In [ ]:
import os

REQUIRED_FILES = {
    "train/train_source1.tsv",
    "train/train_source2.tsv",
    "train/train_source3.tsv",
    "train/train_ground_truth.tsv",
    "test/test_source1.tsv",
    "test/test_source2.tsv",
    "test/test_source3.tsv"
}

KAGGLE_DATA_ROOT = None
possible_roots = []

for root, dirs, files in os.walk("/kaggle/input"):
    if "train_source1.tsv" in files and "train" in root:
        candidate_root = os.path.dirname(root)
        if candidate_root not in possible_roots:
            possible_roots.append(candidate_root)

if len(possible_roots) == 1:
    KAGGLE_DATA_ROOT = possible_roots[0]
    print(f"Automatically discovered data root: {KAGGLE_DATA_ROOT}")
elif len(possible_roots) > 1:
    print("Found multiple possible data roots:")
    for pr in possible_roots:
        print(f"  - {pr}")
    print("Please set KAGGLE_DATA_ROOT manually in the next cell.")
else:
    print("Could not find the dataset! Ensure the Amazon ML Challenge dataset is attached to this notebook.")
    # Fallback to local 'data' directory if running locally
    if os.path.exists("data/raw/train/train_source1.tsv"):
        print("Found local data/raw directory, using it as fallback.")
        KAGGLE_DATA_ROOT = os.path.abspath("data/raw")

## 4. Data Path Validation
Validating the discovered path.

In [ ]:
if KAGGLE_DATA_ROOT:
    all_found = True
    for req in REQUIRED_FILES:
        p = os.path.join(KAGGLE_DATA_ROOT, req)
        if not os.path.exists(p):
            print(f"MISSING REQUIRED FILE: {p}")
            all_found = False
    if all_found:
        print("All required data files found!")
    else:
        raise RuntimeError("Missing required data files. Check dataset attachment.")
else:
    raise RuntimeError("KAGGLE_DATA_ROOT is not set. Please set it manually.")

## 4.5 Data Symlinking & Preprocessing
The repository expects data under `data/raw/...` and builds processed parquet files in `data/processed/...`.
Since Kaggle's `/kaggle/input` is read-only, we symlink the raw data into the repo and run the preprocessing scripts.

In [ ]:
import subprocess
if KAGGLE_DATA_ROOT and KAGGLE_DATA_ROOT != os.path.abspath("data/raw"):
    print("\n--- Creating Local Data Symlinks ---")
    os.makedirs("data/raw", exist_ok=True)
    if not os.path.exists("data/raw/train"):
        os.symlink(os.path.join(KAGGLE_DATA_ROOT, "train"), "data/raw/train")
    if not os.path.exists("data/raw/test"):
        os.symlink(os.path.join(KAGGLE_DATA_ROOT, "test"), "data/raw/test")
    print("Symlinks created.")

    print("\n--- Building Processed Data ---")
    !PYTHONPATH=. python scripts/build_processed_data.py

    print("\n--- Building Folds ---")
    !PYTHONPATH=. python scripts/build_folds.py

    # Override to local data directory which now contains raw/ and processed/
    KAGGLE_DATA_ROOT = os.path.abspath("data")


## 5. Artifact Path & Environment Setup
Setting up output directories and environment variables.

In [ ]:
KAGGLE_ARTIFACT_DIR = "/kaggle/working/artifacts/retrieval/E002-A"
os.makedirs(KAGGLE_ARTIFACT_DIR, exist_ok=True)

os.environ["KAGGLE_DATA_ROOT"] = KAGGLE_DATA_ROOT
os.environ["KAGGLE_ARTIFACT_DIR"] = KAGGLE_ARTIFACT_DIR
os.environ["PYTHONPATH"] = "."

print(f"Output directory set to: {KAGGLE_ARTIFACT_DIR}")

## 6. Environment Verification Script
Running repository GPU checks and verifying RAPIDS libraries.

In [ ]:
!PYTHONPATH=. python scripts/check_gpu_environment.py

print("\n--- Verifying Core Libraries ---")
try:
    import cupy as cp
    print(f"CuPy found (v{cp.__version__})")
    print(f"Detected {cp.cuda.runtime.getDeviceCount()} GPU(s).")
    for i in range(cp.cuda.runtime.getDeviceCount()):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(f"  GPU {i}: {props['name'].decode('utf-8')}")
except ImportError:
    print("CuPy not found!")

try:
    import cuml
    print(f"cuML found (v{cuml.__version__})")
except ImportError:
    print("cuML not found!")
    
try:
    import cudf
    print(f"cuDF found (v{cudf.__version__})")
except ImportError:
    print("cuDF not found!")

## 7. Dependency Handling
Installing only light missing dependencies. Not upgrading RAPIDS components aggressively.

In [ ]:
print("Installing requirements... (avoiding heavy upgrades)")
!pip install -r requirements.txt -q

## 8. Run Tests
Running all tests, and then explicitly running `test_gpu_retrieval.py`.

In [ ]:
print("Running full test suite...")
!PYTHONPATH=. python -m pytest tests/ -q

print("\nRunning GPU retrieval specific tests...")
!PYTHONPATH=. python -m pytest tests/test_gpu_retrieval.py -v

## 9. Display E002 GPU Config
Displaying `configs/e002_word_gpu.yaml` to ensure correctness before running.

In [ ]:
import yaml

with open("configs/e002_word_gpu.yaml", "r") as f:
    e002_config = yaml.safe_load(f)
    
print("--- E002 GPU Config ---")
for k, v in e002_config['retrieval'].items():
    print(f"{k}: {v}")

> [!WARNING]
> **SMOKE TEST PREPARATION**
> We will run a 1000-sample smoke test first against the **full target corpus**.
> This verifies GPU scalability without running a full 50k batch.

In [ ]:
import copy

smoke_config = copy.deepcopy(e002_config)
smoke_config['retrieval']['sample_size'] = 1000

smoke_config_path = "/kaggle/working/e002_gpu_smoke.yaml"
with open(smoke_config_path, "w") as f:
    yaml.dump(smoke_config, f)

print(f"Smoke config created at {smoke_config_path} with sample_size = 1000")

## 10. Run Smoke Test
Pre/Post GPU checks included.

In [ ]:
print("=== PRE-SMOKE GPU STATUS ===")
!nvidia-smi

print("\n=== RUNNING SMOKE TEST ===")
!PYTHONPATH=. python -u scripts/run_e002_word_probe.py --config /kaggle/working/e002_gpu_smoke.yaml

print("\n=== POST-SMOKE GPU STATUS ===")
!nvidia-smi

## 11. Smoke Result Inspection
Checking output artifacts to verify success.

In [ ]:
import json

print(f"Files in {KAGGLE_ARTIFACT_DIR}:")
for f in os.listdir(KAGGLE_ARTIFACT_DIR):
    print(f" - {f}")

metrics_path = os.path.join(KAGGLE_ARTIFACT_DIR, "probe_metrics.json")
if os.path.exists(metrics_path):
    with open(metrics_path, "r") as f:
        metrics = json.load(f)
    print("\nSmoke Test Metrics Generated:")
    print(json.dumps(metrics, indent=2))
else:
    print("\nWARNING: probe_metrics.json not found! Smoke test may have failed.")
    raise RuntimeError("Smoke test failed to produce metrics.")

> [!CAUTION]
> **RUN FULL E002-A 50K PROBE**
> This cell runs the main GPU retrieval probe.
> RUN ONLY AFTER THE SMOKE TEST PASSES.

In [ ]:
print("=== STARTING FULL 50K PROBE ===")
!PYTHONPATH=. python -u scripts/run_e002_word_probe.py --config configs/e002_word_gpu.yaml

## 12. Result Display Cells
Rendering JSON and CSV outputs.

In [ ]:
import pandas as pd

def show_json(filename):
    path = os.path.join(KAGGLE_ARTIFACT_DIR, filename)
    if os.path.exists(path):
        with open(path, "r") as f:
            data = json.load(f)
        print(f"\n--- {filename} ---")
        print(json.dumps(data, indent=2))

show_json("environment.json")
show_json("probe_metrics.json")
show_json("performance.json")

csv_path = os.path.join(KAGGLE_ARTIFACT_DIR, "k_frontier.csv")
if os.path.exists(csv_path):
    print("\n--- k_frontier.csv ---")
    display(pd.read_csv(csv_path))

md_path = os.path.join(KAGGLE_ARTIFACT_DIR, "report.md")
if os.path.exists(md_path):
    print("\n--- report.md (preview) ---")
    with open(md_path, "r") as f:
        print(f.read()[:1000] + "...")

## 13. Final Decision Helper

This notebook itself **DOES NOT** decide the architecture.
The results displayed above will determine if the status becomes:

`E002_GLOBAL_CHAR_TFIDF_GPU_VIABLE`

or:

`E002_GLOBAL_CHAR_TFIDF_REJECTED`

**DO NOT** automatically start E002 or commit these artifacts from Kaggle.

## 14. Package Artifacts
Zip the results for easy download from the Kaggle interface.

In [ ]:
import shutil

zip_path = "/kaggle/working/E002-A-artifacts"
shutil.make_archive(zip_path, 'zip', KAGGLE_ARTIFACT_DIR)
print(f"Artifacts packaged to: {zip_path}.zip")